# LLM Classification Finetuning - Logistic Regression Baseline

This notebook builds a first submission for the competition using a single 3-class classifier:

- `winner_model_a`
- `winner_model_b`
- `winner_tie`

The model uses TF-IDF features from `prompt`, `response_a`, and `response_b`, plus a few simple length features. It is intentionally plain and fast so it can act as a reliable baseline before moving to BERT/LoRA.

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_SPLITS = 3
RUN_CV = False
FAST_MODE = True

## Load Data

The path below works both on Kaggle and in this local project folder.

In [ ]:
def find_data_dir():
    candidates = [
        Path('/kaggle/input/competitions/llm-classification-finetuning'),
        Path('/kaggle/input/llm-classification-finetuning'),
        Path('llm-classification-finetuning'),
        Path('.'),
    ]
    for path in candidates:
        if (path / 'train.csv').exists() and (path / 'test.csv').exists():
            return path
    raise FileNotFoundError('Could not find train.csv and test.csv')

DATA_DIR = find_data_dir()
print(DATA_DIR)

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(train.shape, test.shape)
train.head()

## Parse JSON List Columns

`prompt`, `response_a`, and `response_b` are stored as JSON lists. We join each list into readable text while preserving turn boundaries.

In [ ]:
def parse_json_list(value):
    if pd.isna(value):
        return []
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(x) if x is not None else '' for x in parsed]
        return [str(parsed)]
    except Exception:
        return [str(value)]

def join_turns(value):
    return '\n'.join(parse_json_list(value))

for df in [train, test]:
    df['prompt_text'] = df['prompt'].apply(join_turns)
    df['response_a_text'] = df['response_a'].apply(join_turns)
    df['response_b_text'] = df['response_b'].apply(join_turns)

train[['prompt_text', 'response_a_text', 'response_b_text']].head(2)

## Build Text and Numeric Features

The classifier sees the task as a direct comparison between response A and response B.

In [ ]:
def build_comparison_text(df):
    return (
        'Prompt:\n' + df['prompt_text'].fillna('') +
        '\n\nResponse A:\n' + df['response_a_text'].fillna('') +
        '\n\nResponse B:\n' + df['response_b_text'].fillna('')
    )

def build_numeric_features(df):
    prompt_len = df['prompt_text'].str.len().fillna(0).to_numpy()
    a_len = df['response_a_text'].str.len().fillna(0).to_numpy()
    b_len = df['response_b_text'].str.len().fillna(0).to_numpy()
    a_words = df['response_a_text'].str.split().str.len().fillna(0).to_numpy()
    b_words = df['response_b_text'].str.split().str.len().fillna(0).to_numpy()
    prompt_turns = df['prompt'].apply(lambda x: len(parse_json_list(x))).to_numpy()

    features = np.column_stack([
        prompt_len,
        a_len,
        b_len,
        a_len - b_len,
        np.abs(a_len - b_len),
        a_words,
        b_words,
        a_words - b_words,
        np.abs(a_words - b_words),
        prompt_turns,
    ])
    return features.astype(np.float32)

X_text = build_comparison_text(train)
X_test_text = build_comparison_text(test)

label_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
y = train[label_cols].values.argmax(axis=1)

print(pd.Series(y).value_counts(normalize=True).sort_index())

## Vectorize

Word TF-IDF catches semantic patterns. Character TF-IDF often helps with style, formatting, and small wording differences.

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=80_000 if FAST_MODE else 250_000,
    sublinear_tf=True,
    strip_accents='unicode',
)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=3,
    max_features=80_000 if FAST_MODE else 200_000,
    sublinear_tf=True,
)

numeric_scaler = StandardScaler()

X_word = word_vectorizer.fit_transform(X_text)
X_char = char_vectorizer.fit_transform(X_text)
X_num = csr_matrix(numeric_scaler.fit_transform(build_numeric_features(train)))

X = hstack([X_word, X_char, X_num]).tocsr()

X_test_word = word_vectorizer.transform(X_test_text)
X_test_char = char_vectorizer.transform(X_test_text)
X_test_num = csr_matrix(numeric_scaler.transform(build_numeric_features(test)))
X_test = hstack([X_test_word, X_test_char, X_test_num]).tocsr()

print(X.shape, X_test.shape)
print(f'Total training features: {X.shape[1]:,}')

## Cross-Validation

This estimates local multi-class log loss. Lower is better.

In [ ]:
def make_model():
    return LogisticRegression(
        C=1.0,
        solver='lbfgs' if FAST_MODE else 'saga',
        penalty='l2',
        class_weight=None,
        max_iter=500 if FAST_MODE else 1000,
        tol=1e-3 if FAST_MODE else 1e-4,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1,
    )

if RUN_CV:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(train), 3), dtype=np.float32)
    fold_scores = []

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y), 1):
        model = make_model()
        print(f'Fold {fold}: fitting LogisticRegression with solver={model.solver}, max_iter={model.max_iter}')
        start = time.time()
        model.fit(X[trn_idx], y[trn_idx])
        print(f'Fold {fold}: actual iterations {model.n_iter_}, elapsed {(time.time() - start) / 60:.2f} min')
        val_pred = model.predict_proba(X[val_idx])
        oof[val_idx] = val_pred
        score = log_loss(y[val_idx], val_pred, labels=[0, 1, 2])
        fold_scores.append(score)
        print(f'Fold {fold}: {score:.5f}')

    print(f'CV log loss: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}')

## Train Final Model and Create Submission

In [ ]:
final_model = make_model()
print(f'Fitting final LogisticRegression with solver={final_model.solver}, max_iter={final_model.max_iter}')
print(f'Training matrix shape: {X.shape}')
start = time.time()
final_model.fit(X, y)
print(f'Actual iterations used: {final_model.n_iter_}')
print(f'Final training elapsed: {(time.time() - start) / 60:.2f} min')

test_pred = final_model.predict_proba(X_test)

submission = sample_submission.copy()
submission[label_cols] = test_pred

submission.to_csv('submission.csv', index=False)
submission.head()

## Notes for Next Iteration

After this baseline, useful upgrades are:

1. Try separate vectorizers for prompt, response A, and response B.
2. Add features for markdown/code blocks, refusal language, list counts, and response similarity.
3. Blend several Logistic Regression models with different `C`, n-gram ranges, and seeds.
4. Move to a transformer classifier such as DeBERTa.
5. Ensemble the transformer with this baseline instead of replacing it immediately.